# 🎙️ DEEPFAKE AUDIO DETECTION USING HYBRID CNN-LSTM NEURAL NETWORKS

**Author / Project Team**: Sadhika & Team  
**Frameworks**: TensorFlow / Keras, Librosa, Scikit-learn, Gradio  
**Task**: Binary Classification (Bona Fide Genuine Human Speech vs. Spoofed / AI-Synthesized Voice)  

---

### 📌 Problem Description
Modern generative AI tools (such as neural vocoders, diffusion models, and voice cloning architectures) can produce highly convincing synthetic speech. These synthetic voices can be exploited for voice phishing, CEO fraud, unauthorized biometric impersonation, and identity theft.

This project explores how a **Hybrid Convolutional-Recurrent Deep Learning Architecture (CNN-LSTM)** detects deepfakes by:
1. **Spectral Feature Learning (CNN)**: Learning high-frequency spectral artifacts, phase irregularities, and unnatural vocoder formant profiles from **Log-Mel Spectrograms**.
2. **Sequential Temporal Modeling (LSTM)**: Tracking temporal coherence, phoneme-to-phoneme transition consistency, and vocal tract dynamics across sequential time frames.

---
### ⚡ Google Colab Setup
Make sure GPU acceleration is enabled:
Go to **Runtime** → **Change runtime type** → Select **T4 GPU** (or CPU if GPU is unavailable).

## Step 1: Install Required Libraries

In [ ]:
# Install necessary audio processing, visualization, and interface libraries
!pip install librosa soundfile scikit-learn matplotlib seaborn gradio tqdm -q
print("✅ All dependencies successfully installed!")

## Step 2: Verify Python & GPU Environment

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import soundfile as sf
import tensorflow as tf
import sklearn
import gradio as gr

print("TensorFlow Version:", tf.__version__)
print("Librosa Version:   ", librosa.__version__)
print("NumPy Version:     ", np.__version__)
print("Scikit-learn:      ", sklearn.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 GPU Detected: {gpus[0].name} (Hardware acceleration active)")
else:
    print("ℹ️ No GPU detected. Running on CPU.")

## Step 3: Global Audio & Model Configurations
Standard audio settings matching research benchmarks (ASVspoof 2019):  
- **Sample Rate**: 16,000 Hz (16 kHz mono)  
- **Audio Duration**: 3.0 seconds (48,000 samples)  
- **Mel Bands**: 128 frequency bands  
- **FFT Size**: 1024 samples | **Hop Length**: 512 samples  
- **Target Input Shape**: `(128, 94, 1)`

In [ ]:
SAMPLE_RATE = 16000
DURATION = 3.0
NUM_SAMPLES = int(SAMPLE_RATE * DURATION)  # 48,000 samples

N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
FMIN = 20
FMAX = 8000
TIME_FRAMES = 94
INPUT_SHAPE = (N_MELS, TIME_FRAMES, 1)

BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 0.001
CLASS_NAMES = ["Bona Fide (Genuine)", "Spoof (Deepfake)"]

os.makedirs("data/raw/bonafide", exist_ok=True)
os.makedirs("data/raw/spoof", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("Audio & Model configuration set successfully!")

## Step 4: Dataset Preparation

You have two options for data:
1. **Option A (Instant Demo Dataset)**: Generates genuine speech-like waveforms (with natural pitch vibrato and human vocal tract formants) and spoofed recordings (with vocoder phase distortions and metallic buzz). This lets you run the entire project **immediately without downloading gigabytes of data**.
2. **Option B (Official ASVspoof 2019 LA Dataset)**: Download from University of Edinburgh DataShare or Kaggle and extract into `data/raw/`.

The cell below builds the starter dataset so you can train right away.

In [ ]:
def synthesize_audio_sample(label_type="bonafide", duration=DURATION, sr=SAMPLE_RATE):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    if label_type == "bonafide":
        # Human vocal pitch contour with prosodic drift and vocal tract formants
        base_f0 = np.random.uniform(110.0, 240.0)
        f0_drift = np.sin(2 * np.pi * np.random.uniform(1.5, 4.0) * t) * 8.0
        phase_f0 = 2 * np.pi * np.cumsum(base_f0 + f0_drift) / sr

        f1_mult, f2_mult, f3_mult = np.random.uniform(2.5, 3.5), np.random.uniform(5.0, 7.5), np.random.uniform(9.0, 12.0)
        audio = 1.0 * np.sin(phase_f0) + 0.5 * np.sin(f1_mult * phase_f0) + 0.25 * np.sin(f2_mult * phase_f0) + 0.12 * np.sin(f3_mult * phase_f0)

        # Syllabic articulation envelope
        num_syllables = np.random.randint(4, 8)
        envelope = np.zeros_like(t)
        for i in range(num_syllables):
            center = (i + 0.5) * (duration / num_syllables)
            width = (duration / num_syllables) * 0.4
            envelope += np.exp(-((t - center) ** 2) / (2 * width ** 2))
        audio = audio * (envelope / (np.max(envelope) + 1e-6))
        audio += np.random.normal(0, 0.015, len(t))
    else:
        # Spoofed audio: Vocoder phase distortion, rigid pitch, and synthetic noise
        rigid_f0 = np.random.choice([120.0, 150.0, 180.0, 210.0])
        audio = np.zeros_like(t)
        for h in range(1, 14):
            audio += (1.0 / np.sqrt(h)) * np.random.uniform(0.7, 1.2) * np.sin(2 * np.pi * (rigid_f0 * h) * t)
        
        # Glitches and high-frequency vocoder hiss
        glitches = np.random.choice(len(t), size=int(duration * 6), replace=False)
        audio[glitches] *= np.random.uniform(-1.5, 1.5, size=len(glitches))
        audio += np.random.normal(0, 0.045, len(t)) * (np.sin(2 * np.pi * 3200 * t) ** 2)

    # Normalize amplitude
    max_amp = np.max(np.abs(audio))
    if max_amp > 1e-6:
        audio = audio / max_amp * 0.85
    return audio.astype(np.float32)

# Generate dataset samples
NUM_SAMPLES_PER_CLASS = 60
records = []

print(f"Generating {NUM_SAMPLES_PER_CLASS} Bona Fide (genuine) samples...")
for i in range(NUM_SAMPLES_PER_CLASS):
    audio = synthesize_audio_sample("bonafide")
    p = f"data/raw/bonafide/bonafide_{i:04d}.wav"
    sf.write(p, audio, SAMPLE_RATE)
    records.append({"file_path": p, "label": 0, "class_name": "bonafide"})

print(f"Generating {NUM_SAMPLES_PER_CLASS} Spoof (deepfake) samples...")
for i in range(NUM_SAMPLES_PER_CLASS):
    audio = synthesize_audio_sample("spoof")
    p = f"data/raw/spoof/spoof_{i:04d}.wav"
    sf.write(p, audio, SAMPLE_RATE)
    records.append({"file_path": p, "label": 1, "class_name": "spoof"})

df_dataset = pd.DataFrame(records)
df_dataset.to_csv("data/protocol.csv", index=False)
print(f"\nDataset ready! Total samples: {len(df_dataset)} ({df_dataset['class_name'].value_counts().to_dict()})")

## Step 5: Feature Extraction — Waveforms & Log-Mel Spectrograms
The raw audio is converted into a **Log-Mel Spectrogram**, which projects sound energy into human-perceptual frequency bins over time.

In [ ]:
def load_and_preprocess_audio(file_path, target_sr=SAMPLE_RATE, target_len=NUM_SAMPLES):
    y, sr = librosa.load(file_path, sr=target_sr, mono=True)
    if len(y) < target_len:
        repeats = int(np.ceil(target_len / len(y)))
        y = np.tile(y, repeats)[:target_len]
    elif len(y) > target_len:
        start = (len(y) - target_len) // 2
        y = y[start:start + target_len]
    return y, target_sr

def compute_log_mel_spectrogram(y, sr=SAMPLE_RATE, target_frames=TIME_FRAMES):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    
    # Ensure exact time dimensions
    if log_mel.shape[1] < target_frames:
        log_mel = np.pad(log_mel, ((0, 0), (0, target_frames - log_mel.shape[1])), mode='edge')
    else:
        log_mel = log_mel[:, :target_frames]
        
    # Min-Max normalization to [0, 1]
    m_min, m_max = log_mel.min(), log_mel.max()
    if m_max - m_min > 1e-6:
        norm_mel = (log_mel - m_min) / (m_max - m_min)
    else:
        norm_mel = np.zeros_like(log_mel)
        
    return np.expand_dims(norm_mel.astype(np.float32), axis=-1)

# Visual comparison: Bona Fide vs Spoof Spectrograms
y_bona, _ = load_and_preprocess_audio("data/raw/bonafide/bonafide_0000.wav")
spec_bona = compute_log_mel_spectrogram(y_bona)[:, :, 0]

y_spoof, _ = load_and_preprocess_audio("data/raw/spoof/spoof_0000.wav")
spec_spoof = compute_log_mel_spectrogram(y_spoof)[:, :, 0]

fig, axes = plt.subplots(2, 2, figsize=(15, 7))

axes[0, 0].plot(np.linspace(0, 3, len(y_bona)), y_bona, color="#16a34a")
axes[0, 0].set_title("Bona Fide (Genuine) Waveform", fontweight="bold")
axes[0, 0].set_ylabel("Amplitude")

axes[0, 1].plot(np.linspace(0, 3, len(y_spoof)), y_spoof, color="#dc2626")
axes[0, 1].set_title("Spoof (Deepfake) Waveform", fontweight="bold")
axes[0, 1].set_ylabel("Amplitude")

im1 = axes[1, 0].imshow(spec_bona, origin="lower", aspect="auto", cmap="magma", extent=[0, 3, FMIN, FMAX])
axes[1, 0].set_title("Bona Fide Mel-Spectrogram (Natural Harmonics)", fontweight="bold")
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].set_ylabel("Frequency (Hz)")
fig.colorbar(im1, ax=axes[1, 0], format="%+2.0f dB")

im2 = axes[1, 1].imshow(spec_spoof, origin="lower", aspect="auto", cmap="magma", extent=[0, 3, FMIN, FMAX])
axes[1, 1].set_title("Spoof Mel-Spectrogram (Vocoder / Static Harmonics)", fontweight="bold")
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].set_ylabel("Frequency (Hz)")
fig.colorbar(im2, ax=axes[1, 1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

## Step 6: Dataset Generator & Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_dataset, test_size=0.2, stratify=df_dataset["label"], random_state=42
)
print(f"Training set: {len(train_df)} samples | Validation set: {len(val_df)} samples")

class AudioDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE, shuffle=True):
        self.df = df.copy().reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch = self.df.iloc[batch_idx]
        X, y = [], []
        for _, row in batch.iterrows():
            audio, sr = load_and_preprocess_audio(row["file_path"])
            spec = compute_log_mel_spectrogram(audio, sr)
            X.append(spec)
            y.append(row["label"])
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

train_gen = AudioDataGenerator(train_df, batch_size=BATCH_SIZE, shuffle=True)
val_gen = AudioDataGenerator(val_df, batch_size=BATCH_SIZE, shuffle=False)
print("Data generators initialized!")

## Step 7: Build the Hybrid CNN-LSTM Architecture

### Architecture Explanation:
- **2D CNN Front-End**: 3 Convolutional blocks capture spatial spectral patterns (formant bands, high-frequency anomalies).
- **Reshape / Permute**: Transforms the output from `(batch, freq, time, channels)` to `(batch, time, features)`.
- **Bidirectional LSTM Back-End**: 2 stacked BiLSTM layers process sequential acoustic changes forward and backward in time.
- **Dense Classifier**: Fully connected layers with dropout and a Sigmoid activation outputting the probability that an audio clip is a deepfake.

In [ ]:
from tensorflow.keras import layers, models, optimizers

def build_hybrid_cnn_lstm(input_shape=INPUT_SHAPE, lr=LEARNING_RATE):
    inputs = layers.Input(shape=input_shape, name="spectrogram_input")
    
    # --- CNN Feature Extractor ---
    x = layers.Conv2D(32, (3, 3), padding="same", name="conv1")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)  # (64, 47, 32)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, (3, 3), padding="same", name="conv2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 1))(x)  # (32, 47, 64)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3, 3), padding="same", name="conv3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 1))(x)  # (16, 47, 128)
    x = layers.Dropout(0.25)(x)

    # --- Reshape to Sequential (Time, Features) ---
    # Permute from (batch, freq, time, channels) -> (batch, time, freq, channels)
    x = layers.Permute((2, 1, 3))(x)
    time_steps = x.shape[1]   # 47 frames
    feat_dim = x.shape[2] * x.shape[3]  # 16 * 128 = 2048
    x = layers.Reshape((time_steps, feat_dim))(x)

    # --- Bidirectional LSTM Temporal Processor ---
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True), name="bilstm1")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=False), name="bilstm2")(x)
    x = layers.Dropout(0.3)(x)

    # --- Dense Classification Head ---
    x = layers.Dense(64, activation="relu", name="dense_fc")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="spoof_probability")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="CNN_LSTM_Deepfake_Detector")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
    )
    return model

model = build_hybrid_cnn_lstm()
model.summary()

## Step 8: Train the Model with Smart Callbacks

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="models/best_cnn_lstm_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Starting model training...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

model.save("models/final_cnn_lstm_model.keras")
print("Training complete! Saved to models/best_cnn_lstm_model.keras")

## Step 9: Training History & Learning Curves

In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Loss Plot
axes[0].plot(epochs_range, history.history["loss"], label="Training Loss", color="#dc2626", lw=2)
axes[0].plot(epochs_range, history.history["val_loss"], label="Validation Loss", color="#ea580c", linestyle="--", lw=2)
axes[0].set_title("Training & Validation Loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Crossentropy Loss")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# Accuracy Plot
axes[1].plot(epochs_range, history.history["accuracy"], label="Training Accuracy", color="#2563eb", lw=2)
axes[1].plot(epochs_range, history.history["val_accuracy"], label="Validation Accuracy", color="#0284c7", linestyle="--", lw=2)
axes[1].set_title("Training & Validation Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("reports/training_curves.png", dpi=200)
plt.show()

## Step 10: Performance Evaluation (Metrics, Confusion Matrix, ROC-AUC, EER)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Predict on validation set
y_true, y_scores = [], []
for i in range(len(val_gen)):
    xb, yb = val_gen[i]
    preds = model.predict(xb, verbose=0).flatten()
    y_scores.extend(preds)
    y_true.extend(yb)

y_true = np.array(y_true)
y_scores = np.array(y_scores)
y_pred = (y_scores >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_scores)

# Compute Equal Error Rate (EER)
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0

print("=" * 45)
print(f"Validation Accuracy:  {acc * 100:.2f}%")
print(f"Precision:            {prec * 100:.2f}%")
print(f"Recall:               {rec * 100:.2f}%")
print(f"F1-Score:             {f1 * 100:.2f}%")
print(f"ROC-AUC:              {roc_auc:.4f}")
print(f"Equal Error Rate (EER): {eer * 100:.2f}%")
print("=" * 45)
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix & ROC Curve Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Confusion Matrix", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Ground Truth")

# ROC Curve
axes[1].plot(fpr, tpr, color="#2563eb", lw=2, label=f"ROC (AUC = {roc_auc:.4f})")
axes[1].plot([0, 1], [0, 1], "--", color="#9ca3af")
axes[1].scatter([eer], [1 - eer], color="#dc2626", s=50, label=f"EER = {eer * 100:.2f}%")
axes[1].set_title("Receiver Operating Characteristic (ROC)", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("reports/evaluation_plots.png", dpi=200)
plt.show()

## Step 11: Single Audio File Inference Function
Test any audio file and inspect the probability that it was generated by an AI model.

In [ ]:
def predict_audio_file(file_path):
    audio, sr = load_and_preprocess_audio(file_path)
    spec = compute_log_mel_spectrogram(audio, sr)
    batch_input = np.expand_dims(spec, axis=0)
    
    p_spoof = float(model.predict(batch_input, verbose=0)[0][0])
    p_bona = 1.0 - p_spoof
    is_spoof = p_spoof >= 0.5
    
    label = "Spoof (Deepfake)" if is_spoof else "Bona Fide (Genuine)"
    confidence = (p_spoof if is_spoof else p_bona) * 100.0
    
    print(f"\n{'=' * 40}")
    print(f"File:        {os.path.basename(file_path)}")
    print(f"Verdict:     {label}")
    print(f"Confidence:  {confidence:.2f}%")
    print(f"P(Deepfake): {p_spoof:.4f} | P(Genuine): {p_bona:.4f}")
    print(f"{'=' * 40}\n")
    return label, confidence

# Test on a genuine sample
predict_audio_file("data/raw/bonafide/bonafide_0001.wav")

# Test on a deepfake sample
predict_audio_file("data/raw/spoof/spoof_0001.wav")

## Step 12: Interactive Web Interface (Gradio)
Launch an interactive dashboard directly inside Google Colab!
Set `share=True` to generate a public shareable URL that you can demonstrate to others.

In [ ]:
def analyze_audio_colab(audio_path):
    if audio_path is None:
        return "Please upload or record audio.", None, None
    
    audio, sr = load_and_preprocess_audio(audio_path)
    spec = compute_log_mel_spectrogram(audio, sr)
    batch_input = np.expand_dims(spec, axis=0)
    
    p_spoof = float(model.predict(batch_input, verbose=0)[0][0])
    p_bona = 1.0 - p_spoof
    is_spoof = p_spoof >= 0.5
    
    verdict = "Spoof (Deepfake)" if is_spoof else "Bona Fide (Genuine)"
    color = "#dc2626" if is_spoof else "#16a34a"
    conf = (p_spoof if is_spoof else p_bona) * 100.0
    
    html = f"""
    <div style='padding: 16px; border-radius: 8px; border: 2px solid {color}; background-color: #f8fafc;'>
        <h2 style='color: {color}; margin: 0;'>Verdict: {verdict}</h2>
        <h3 style='margin: 8px 0;'>Confidence: {conf:.2f}%</h3>
        <p>P(Deepfake) = {p_spoof:.4f} | P(Genuine) = {p_bona:.4f}</p>
    </div>
    """
    
    # Waveform and spectrogram plot
    fig, axes = plt.subplots(2, 1, figsize=(8, 4), gridspec_kw={'height_ratios': [1, 1.3]})
    t = np.linspace(0, 3, len(audio))
    axes[0].plot(t, audio, color="#2563eb", lw=0.8)
    axes[0].set_title("Waveform")
    
    im = axes[1].imshow(spec[:, :, 0], origin="lower", aspect="auto", cmap="magma", extent=[0, 3, FMIN, FMAX])
    axes[1].set_title("Log-Mel Spectrogram")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Freq (Hz)")
    plt.tight_layout()
    
    probs = {"Bona Fide (Genuine)": p_bona, "Spoof (Deepfake)": p_spoof}
    return html, probs, fig

demo = gr.Interface(
    fn=analyze_audio_colab,
    inputs=gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload Audio (.wav, .mp3)"),
    outputs=[
        gr.HTML(label="Prediction Result"),
        gr.Label(label="Class Probabilities"),
        gr.Plot(label="Spectrogram Inspection")
    ],
    title="🎙️ Deepfake Audio Detector | CNN-LSTM",
    description="Upload or record human speech to determine whether it is genuine or an AI-generated synthetic voice."
)

# Launch inline in Colab and generate shareable public link
demo.launch(share=True, debug=False)